# Exercise 04 (optional) — LSTM vs. GRU

In the main notebook the LSTM and the GRU ended in a tie: on AG News they were within half a point of macro-F1 of each other. That is a typical result on real data — and it makes it hard to see what actually distinguishes the two.

In this notebook you compare them on a task that isolates the one thing both were designed for: **remembering something for many time steps**. The task is synthetic and the models are tiny, so **everything runs in a minute or two on a CPU** and there is nothing to download.

## What you will do
1. **The recall task** — generate sequences where the only thing that matters is the *first* token.
2. **The models** — one small classifier that can use an LSTM or a GRU.
3. **LSTM vs. GRU** — train both for increasing distances and plot how far back each one can remember.
4. **Why?** — look inside the gates at initialisation and explain the result from the update equations.
5. **The fix** — change one line of initialisation and run the comparison again.
6. **What does it cost?** — compare parameters and time per training step.
7. **Optional: the same trick for the GRU.**

## How to work through it
Same as before: run the cells **in order**, fill in every `# TODO`, and make sure each **✅ Check** cell passes before moving on. Tasks that say *Your answer here* want a short written answer.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. The recall task

Every example is a sequence of `seq_len` token ids:

- the **first** token is one of `NUM_SYMBOLS = 8` *symbols* (ids 1–8) — this is what the model has to remember,
- **all other** tokens are random *noise* tokens (ids 9–16) that carry no information at all.

After reading the whole sequence, the model has to say **which symbol came first**:

```
  x = [ 3, 12, 9, 16, 11, 14, 9, 10, 13, 12 ]      ->      y = 2   (symbol id 3 -> class 2)
        ^  └──────────── noise ────────────┘
      symbol
```

There is nothing to learn about language here. The only difficulty is carrying one piece of information across `seq_len - 1` steps of distraction, so `seq_len` is a dial that sets how long the dependency is. Because we can generate as many examples as we like, every training batch is fresh: there is no training set, no test set and no overfitting.

**1.1 Complete `make_batch`.**

Hints:
- `torch.randint(low, high, size)` draws integers from `low` (inclusive) to `high` (**exclusive**).
- Draw the labels `y` as classes `0..7`; the matching symbol id is `y + 1` (id 0 is left unused, as the padding id was in the main notebook).
- `x[:, 0] = ...` overwrites the first position of every sequence in the batch.

In [ ]:
NUM_SYMBOLS = 8                         # symbols to remember: ids 1..8
NUM_NOISE = 8                           # noise tokens:        ids 9..16
VOCAB_SIZE = 1 + NUM_SYMBOLS + NUM_NOISE


def make_batch(batch_size, seq_len):
    # Return (x, y): x is (batch_size, seq_len) token ids, y is (batch_size,) classes in 0..NUM_SYMBOLS-1.
    x = ...  # TODO: (batch_size, seq_len) random *noise* ids, i.e. integers in 9..16
    y = ...  # TODO: (batch_size,) random classes in 0..7
    ...      # TODO: put the symbol id that belongs to each class at the first position of its sequence
    return x, y


x, y = make_batch(3, 10)
print(x)
print(y)

In [ ]:
# ✅ Check your data
x, y = make_batch(512, 25)
assert x.shape == (512, 25) and y.shape == (512,), "wrong shapes"
assert x.dtype == torch.long and y.dtype == torch.long, "ids and classes must be long tensors"
assert torch.equal(x[:, 0], y + 1), "the first token must be the symbol id of the class: y + 1"
assert x[:, 1:].min() == 1 + NUM_SYMBOLS and x[:, 1:].max() == VOCAB_SIZE - 1, "all other tokens must be noise ids 9..16"
assert y.min() == 0 and y.max() == NUM_SYMBOLS - 1, "classes must cover 0..7"
print("Looks good ✅")

**1.2 What accuracy does a model get that has learned nothing** — or that has *forgotten* the first token by the time it reaches the end? Keep this number in mind, you will see it a lot.

---

*Your answer here:*

---

## 2. The models

The classifier has the same three parts as in the main notebook — embedding → recurrent layer → linear layer on the final hidden state — only much smaller. To avoid writing the class twice, the recurrent layer is chosen by name.

**2.1 Complete `RecallModel`.**

Hints:
- `RECURRENT_LAYERS[kind]` is the *class* (`nn.LSTM` or `nn.GRU`); call it like you did before, with `batch_first=True`.
- This time take the last time step of `output` (you showed in 3.2 of the main notebook that it is the final hidden state). This way `forward` is identical for both layers, even though the LSTM returns `(hidden, cell)` and the GRU only `hidden` as the second value.

In [ ]:
EMBED_DIM = 16
HIDDEN_DIM = 32

RECURRENT_LAYERS = {"LSTM": nn.LSTM, "GRU": nn.GRU}


class RecallModel(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.embedding = ...  # TODO: nn.Embedding for VOCAB_SIZE ids of dimension EMBED_DIM
        self.rec = ...        # TODO: the recurrent layer called `kind`, EMBED_DIM -> HIDDEN_DIM, batch_first=True
        self.fc = ...         # TODO: nn.Linear from HIDDEN_DIM to NUM_SYMBOLS

    def forward(self, x):
        output, _ = ...       # TODO: embedding -> recurrent layer; output is (B, L, H)
        return ...            # TODO: linear layer on the last time step of output -> (B, NUM_SYMBOLS)


def count_parameters(module):
    return sum(p.numel() for p in module.parameters())


for kind in RECURRENT_LAYERS:
    model = RecallModel(kind)
    print(f"{kind:<5} recurrent layer: {count_parameters(model.rec):,} parameters   (whole model: {count_parameters(model):,})")

In [ ]:
# ✅ Check your model
x, y = make_batch(4, 12)
for kind in RECURRENT_LAYERS:
    model = RecallModel(kind)
    assert isinstance(model.rec, RECURRENT_LAYERS[kind]), f"model.rec should be an nn.{kind}"
    assert model(x).shape == (4, NUM_SYMBOLS), f"{kind}: expected logits of shape (4, {NUM_SYMBOLS})"
assert count_parameters(RecallModel("LSTM").rec) == 6_400, "the LSTM layer should have 6,400 parameters"
assert count_parameters(RecallModel("GRU").rec) == 4_800, "the GRU layer should have 4,800 parameters"
print("Looks good ✅")

## 3. LSTM vs. GRU

### Training

The training loop is the one from the main notebook (Adam, gradient clipping), with two differences: every step draws a **fresh batch** from `make_batch`, and we **stop early** as soon as the model has solved the task, which we define as more than 95% accuracy on 1,000 fresh examples.

**3.1 Complete `accuracy` and the training step in `train_recall`.**

In [ ]:
BATCH_SIZE = 128


def accuracy(model, seq_len, n=1000):
    # Fraction of n fresh examples of length seq_len that the model classifies correctly.
    model.eval()
    with torch.no_grad():
        x, y = make_batch(n, seq_len)
        predicted = ...  # TODO: the class with the highest logit, shape (n,)
        return ...       # TODO: fraction of correct predictions, as a Python float  (hint: .float().mean().item())


def train_recall(model, seq_len, max_steps=2500, lr=3e-3):
    # Train until the task is solved (accuracy > 95%) or max_steps is reached.
    # Returns (final accuracy, the step at which the task was solved or None).
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for step in range(1, max_steps + 1):
        model.train()
        x, y = make_batch(BATCH_SIZE, seq_len)

        # TODO: forward pass and loss
        loss = ...

        # TODO: zero the gradients and backpropagate
        ...

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # TODO: update the weights
        ...

        if step % 50 == 0 and accuracy(model, seq_len) > 0.95:
            return accuracy(model, seq_len, n=2000), step

    return accuracy(model, seq_len, n=2000), None


def run_experiment(kind, seq_len, init_fn=None, seed=0):
    # Train a fresh model of the given kind on sequences of length seq_len.
    # `init_fn(model)` can be used to change the initialisation before training (Part 5).
    torch.manual_seed(seed)
    model = RecallModel(kind)
    if init_fn is not None:
        init_fn(model)
    acc, solved_at = train_recall(model, seq_len)
    outcome = f"solved after {solved_at:>4} steps" if solved_at else "not solved"
    print(f"  seq_len {seq_len:>3}: accuracy {acc:.3f}   {outcome}")
    return acc, solved_at

In [ ]:
# ✅ Check your training on the easiest setting: remember the symbol for 5 steps
print("LSTM")
acc, solved_at = run_experiment("LSTM", 5)
assert acc > 0.95 and solved_at is not None, "the LSTM should solve seq_len = 5 within a few hundred steps"
assert 0.05 < accuracy(RecallModel("LSTM"), 5) < 0.25, "an untrained model should be at chance level (~0.125)"
print("Looks good ✅")

### The experiment

**3.2 Run the cell below.** It trains a fresh LSTM and a fresh GRU for each distance in `SEQ_LENS` — same seed, same optimiser, same budget of 2,500 steps. It takes about a minute; the runs that *fail* are the slow ones, because they use up the whole budget.

In [ ]:
SEQ_LENS = [10, 20, 30, 40]

results = {}
for kind in ["LSTM", "GRU"]:
    print(kind)
    results[kind] = [run_experiment(kind, seq_len) for seq_len in SEQ_LENS]

**3.3 Plot the final accuracy of both models against the sequence length** in the left panel, and the number of steps each model needed in the right panel.

Hints:
- `results[kind]` is a list of `(accuracy, solved_at)` pairs, one per entry of `SEQ_LENS`.
- `solved_at` is `None` for the runs that failed — plot only the runs that succeeded in the right panel.
- A horizontal line at chance level (`axhline`) makes the left panel much easier to read.

In [ ]:
def plot_results(results):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    for name, runs in results.items():
        # TODO: left panel  — accuracy against SEQ_LENS
        # TODO: right panel — solved_at against seq_len, only for the runs that were solved
        ...

    # TODO: chance level in the left panel, titles, axis labels, legends

    plt.tight_layout()
    plt.show()


plot_results(results)

**3.4 Describe what you see.** Up to which distance does each model solve the task? What does the accuracy look like when a model fails — a gradual decline, or something else? And based on this plot alone, which architecture would you call "better at long-term memory"?

---

*Your answer here:*

---

## 4. Why? A look inside the gates

The LSTM was *invented* to remember things for a long time, so this result should make you suspicious. The place to look is the update of its cell state:

\begin{align}
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t, \qquad f_t = \sigma\left(W_{if}\, x_t + b_{if} + W_{hf}\, h_{t-1} + b_{hf}\right)
\end{align}

Whatever was written into the cell at step 1 is multiplied by the **forget gate** $f_t$ at *every* later step. What survives after $T$ steps is $f_2 \cdot f_3 \cdots f_T$ of it — and the gradient that tells the model "the first token mattered" is scaled by the same product on its way back.

So: **what is $f_t$ in a freshly initialised `nn.LSTM`?** PyTorch initialises all weights and biases of the layer with small random numbers around zero. With small weights, the gate is roughly the sigmoid of its two biases.

PyTorch stores the biases of all four gates stacked in one vector of length `4 * HIDDEN_DIM`, in the order **input, forget, cell candidate, output**:

```
lstm.bias_ih_l0 = [ b_ii | b_if | b_ig | b_io ]        lstm.bias_hh_l0 = [ b_hi | b_hf | b_hg | b_ho ]
                     0:H   H:2H  2H:3H  3H:4H
```

**4.1 Compute the forget gate of a freshly initialised LSTM, and how much of the first token's information survives `T` steps.**

In [ ]:
torch.manual_seed(0)
lstm = RecallModel("LSTM").rec
H = HIDDEN_DIM

with torch.no_grad():
    forget_bias = ...  # TODO: b_if + b_hf, the forget-gate slices of lstm.bias_ih_l0 and lstm.bias_hh_l0, shape (H,)
    forget_gate = ...  # TODO: the sigmoid of it, shape (H,)

f = forget_gate.mean().item()
print(f"Forget gate at initialisation: {f:.3f} on average   (min {forget_gate.min():.3f}, max {forget_gate.max():.3f})\n")

for seq_len in SEQ_LENS:
    surviving = ...  # TODO: the fraction of the cell state written at step 1 that is left after the remaining seq_len - 1 steps
    print(f"  seq_len {seq_len:>3}: {surviving:.1e} of the cell state survives")

In [ ]:
# ✅ Check your numbers
assert forget_gate.shape == (HIDDEN_DIM,), "forget_gate should have one value per hidden unit"
assert 0.4 < f < 0.6, "at initialisation the forget gate should be close to 0.5"
assert surviving < 1e-9, "at seq_len = 40 practically nothing should survive — did you raise f to the power seq_len - 1?"
print("Looks good ✅")

**4.2 Explain the result of Part 3 with these numbers.** Why can the default LSTM not even *start* learning at `seq_len = 20`, although an LSTM with the right weights could solve the task perfectly? (Think about what the gradient with respect to the first token's embedding looks like at initialisation.)

---

*Your answer here:*

---

## 5. The fix: initialise the forget gate to *remember*

If the problem is that the forget gate starts at 0.5, start it somewhere else. Setting the forget-gate bias to a positive value $b$ makes the gate start at $\sigma(b)$ — the LSTM then *remembers by default* and has to learn when to forget, instead of the other way round. This is a well-known trick (Gers et al., 2000; Jozefowicz et al., 2015, who found that it closes the gap between the LSTM and the GRU on several tasks), and some libraries do it by default — PyTorch does not.

**5.1 Complete `set_forget_bias`.**

Hints:
- The gate computes $\sigma(\ldots + b_{if} + b_{hf})$, so the two biases are simply added. Set the forget-gate slice of `bias_ih_l0` to `value` and the forget-gate slice of `bias_hh_l0` to zero, so that the total is exactly `value`.
- Parameters have to be modified inside `torch.no_grad()`. `tensor[a:b].fill_(v)` overwrites a slice in place.
- The layout of the bias vector is the one from Part 4.

In [ ]:
def set_forget_bias(model, value=3.0):
    # Initialise the forget-gate bias of model.rec (an nn.LSTM) so that b_if + b_hf = value.
    lstm, H = model.rec, HIDDEN_DIM
    with torch.no_grad():
        ...  # TODO: forget-gate slice of lstm.bias_ih_l0 -> value
        ...  # TODO: forget-gate slice of lstm.bias_hh_l0 -> 0

In [ ]:
# ✅ Check your initialisation
torch.manual_seed(0)
model = RecallModel("LSTM")
before = [p.clone() for p in model.parameters()]
set_forget_bias(model, 3.0)

H = HIDDEN_DIM
total_bias = model.rec.bias_ih_l0 + model.rec.bias_hh_l0
assert torch.allclose(total_bias[H:2 * H], torch.full((H,), 3.0)), "b_if + b_hf should be exactly 3 for every hidden unit"
changed = sum((p != q).sum().item() for p, q in zip(model.parameters(), before))
assert changed <= 2 * H, "only the forget-gate biases should change — check your slices"
assert torch.equal(model.rec.bias_ih_l0[:H], before[3][:H]), "the input gate (first slice) must be left alone"

f_new = torch.sigmoid(torch.tensor(3.0)).item()
print(f"Forget gate at initialisation: {f_new:.3f}")
for seq_len in SEQ_LENS:
    print(f"  seq_len {seq_len:>3}: {f_new ** (seq_len - 1):.2f} of the cell state survives")
print("Looks good ✅")

**5.2 Run the comparison again**, now with a third contestant: the same LSTM, with the forget-gate bias initialised to 3. Nothing else changes — same architecture, same number of parameters, same seed, same training.

In [ ]:
print("LSTM (forget bias 3)")
results["LSTM (forget bias 3)"] = [run_experiment("LSTM", seq_len, init_fn=set_forget_bias) for seq_len in SEQ_LENS]

plot_results(results)

In [ ]:
# ✅ Check the result
acc, solved_at = results["LSTM (forget bias 3)"][-1]
assert solved_at is not None, f"with the forget bias at 3 the LSTM should solve seq_len = {SEQ_LENS[-1]}"
print("Looks good ✅")

**5.3 Interpret the result.** How does the ranking from 3.4 change? Was the gap you saw in Part 3 a difference between the *architectures*? And what does that tell you about benchmark tables that report "LSTM: x%, GRU: y%"?

---

*Your answer here:*

---

## 6. What does it cost?

Accuracy is one side of the comparison. The GRU's selling point is that it is the *lighter* of the two: three blocks of weights instead of four.

**6.1 Run the cell below** to measure the time of one training step (forward + backward + update) for both models at `seq_len = 40`.

In [ ]:
def time_per_step(kind, seq_len=40, steps=100):
    # Average wall-clock time of one training step, in milliseconds.
    torch.manual_seed(0)
    model = RecallModel(kind)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

    def one_step():
        x, y = make_batch(BATCH_SIZE, seq_len)
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    for _ in range(10):  # warm-up
        one_step()
    start = time.perf_counter()
    for _ in range(steps):
        one_step()
    return 1000 * (time.perf_counter() - start) / steps


print(f"{'':<6}{'recurrent parameters':>22}{'ms per step':>14}")
for kind in RECURRENT_LAYERS:
    n_params = count_parameters(RecallModel(kind).rec)
    print(f"{kind:<6}{n_params:>22,}{time_per_step(kind):>14.1f}")

**6.2 The GRU has 25% fewer parameters than the LSTM. Is it 25% faster on your machine?** Compare with your neighbours. What does this tell you about using the parameter count as a proxy for speed?

---

*Your answer here:*

---

## 7. Optional: the same trick for the GRU

The GRU failed too, just later. It has no forget gate, but its **update gate** $z_t$ plays the same role. In PyTorch's formulation

\begin{align}
h_t = (1 - z_t) \odot \tilde{h}_t + z_t \odot h_{t-1}
\end{align}

so $z_t \approx 1$ means *keep the old state*. (Careful: many textbooks and the lecture slides may define $z_t$ the other way round, with $z_t \approx 1$ meaning *overwrite*. Always check which convention your library uses.) The GRU's biases are stacked in the order **reset, update, candidate**.

**7.1 Write `set_update_bias`, the GRU counterpart of `set_forget_bias`, and add a fourth line to the plot.** Does it extend the GRU's memory in the same way?

In [ ]:
def set_update_bias(model, value=3.0):
    # Initialise the update-gate bias of model.rec (an nn.GRU) so that b_iz + b_hz = value.
    ...  # TODO


# TODO: run the experiment for the GRU with init_fn=set_update_bias, store it in `results`, and plot

---

*Your answer here:*

---

## Take-aways

- The LSTM and the GRU are built on the same idea: a state that is carried along *additively* and gates that decide what to keep. Their capabilities are very similar, which is why they tie so often on real tasks.
- When you *do* see a gap between them, check the boring explanations — initialisation, learning rate, training budget, random seed — before crediting the architecture.
- Fewer parameters does not automatically mean faster. Measure.

*References: Gers, Schmidhuber & Cummins (2000), "Learning to Forget: Continual Prediction with LSTM". Jozefowicz, Zaremba & Sutskever (2015), "An Empirical Exploration of Recurrent Network Architectures". Chung et al. (2014), "Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling".*